# Analyzing long time (averaged prevalence) for different values of $\epsilon_1$ and $\epsilon_2$ in the SIRCMW model

In the SRICmw model, we have two parameters $\epsilon_1$ and $\epsilon_2$ that control the relation between the waning and mutation effects through terms like 
$$
(1+\epsilon_1 SI) \quad \text{and} \quad (1+\epsilon_2 SI).
$$
Note that this perturbation approach allows for small (in the sense explained below) negative values of $\epsilon_1$ and $\epsilon_2$.

This means that if $|{\epsilon_1 SI}|\ll 1$ and $|\epsilon_2 SI|\ll 1$, then the effects of mutation are much smaller than the effects of waning, and we are close to the SIRC model. On the other hand, if $|\epsilon_1 SI|$ and $|\epsilon_2 SI|$ are of order 1 or larger, then the effects of mutation are comparable to or larger than the effects of waning, and we are in a regime where mutation can significantly affect the dynamics.

Of course, depending on the values of $S$ and $I$, we may need to take extremely large values of $\epsilon_1$ and $\epsilon_2$ to reach the regime where mutation dominates (you have already seen this). To roughly normalize the values of $\epsilon_1$ and $\epsilon_2$, we can define then in terms of relative parameters as follows:
$$
\epsilon_1 =\frac{\tilde{\epsilon}_1}{SI(t=0)}, \quad \epsilon_2 = \frac{\tilde{\epsilon}_2}{SI(t=0)}.
$$
This way, we expect to see changes in the dynamics playing around with paremeters $\tilde{\epsilon}_1$ and $\tilde{\epsilon}_2$ that are not that far away from $(0,0)$.

In [ ]:
#Import stuff
import numpy as np
from scipy.optimize import fsolve
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from scipy.interpolate import griddata
from scipy.interpolate import RegularGridInterpolator


In [ ]:
#Read data
prevalence_results = np.load("prevalence_results.npz")
list_eps1 = prevalence_results['list_eps1']
list_eps2 = prevalence_results['list_eps2']
prevalences = prevalence_results['prevalences']
reseed_counts = prevalence_results['reseed_counts']

In [ ]:
#Plot both together
fig, axs = plt.subplots(2, 2, figsize=(16, 12))
axs = axs.flatten()
# Prevalence plot
im1 = axs[0].pcolormesh(list_eps1, list_eps2, prevalences, cmap='turbo', shading='auto')
axs[0].set_xlabel(r'$\tilde{\epsilon}_1$')
axs[0].set_ylabel(r'$\tilde{\epsilon}_2$')
axs[0].set_title('Average Prevalence I(t) over last 10 years')
fig.colorbar(im1, ax=axs[0], label='Average Prevalence I(t)')
# Reseed count plot
im2 = axs[1].pcolormesh(list_eps1, list_eps2, reseed_counts, cmap='inferno', shading='auto')
axs[1].set_xlabel(r'$\tilde{\epsilon}_1$')
axs[1].set_ylabel(r'$\tilde{\epsilon}_2$')
axs[1].set_title('Number of Reseeds')
fig.colorbar(im2, ax=axs[1], label='Number of Reseeds')

#More detail on the low-prevalence region
pcm = axs[2].contourf(list_eps1, list_eps2, prevalences, levels=np.logspace(-3.5, -2., 20), cmap='turbo')
fig.colorbar(pcm, ax=axs[2], label='Average Prevalence I(t)')
axs[2].set_xlabel(r'$\tilde{\epsilon}_1$')
axs[2].set_ylabel(r'$\tilde{\epsilon}_2$')
axs[2].set_title('Zoomed Low Prevalence')

#More detail on the high-prevalence region
pcm = axs[3].contourf(list_eps1, list_eps2, prevalences, levels=np.linspace(0.01, 0.6, 20), cmap='turbo')
fig.colorbar(pcm, ax=axs[3], label='Average Prevalence I(t)')
axs[3].set_xlabel(r'$\tilde{\epsilon}_1$')
axs[3].set_ylabel(r'$\tilde{\epsilon}_2$')
axs[3].set_title('Zoomed High Prevalence')

The axis at the top-left indicates the average prevalence of infection averaged over the last 10 years of the 100-year simulation. We already see some interesting behavior here. There seems to be a region (non-trivially delimited) where the prevalence suddenly jumps depending on the strenght of the mutation effects. To better get a better resolution, we plot in the axis at the bottom-left and bottom-right the prevalence, zooming in low-prevalence 
and high-prevalence regions, respectively. Something intersting is that there seeem to be level-curves along which the prevalence is constant, and these level curves are not straight lines, it would be nice to undersand the shape of these level curves better (even better, characterize them analytically).

The axis at the top-right indicates the number of times the system was reseeded with a new infection. The reseeding is necessary whenever the number of infected individuals drops below a certain threshold (we reach a numerical disease-free equilibrium, even though we should not get there mathematically). The direction of the reseeding follows an unstable eigenvector at the disease-free equilibrium. The reseeding introduces some noise that make the long-term prevalence fluctuate, but we still get a general idea of the behavior, especially if we extrapolate information from the regions where the number of reseeds is low or zero.





### Plotting slices of the surface

This gives us a better idea of the overall beahvior

In [ ]:
interp = RegularGridInterpolator(
    (list_eps1, list_eps2),
    prevalences,
    method='linear',        
    bounds_error=False,
    fill_value=np.nan, 
)
eps1_a, eps2_a = -0.5, -0.5
eps1_b, eps2_b = 2.5, 2.5

# Define the line by two endpoints in (eps1, eps2) space
p_start = np.array([eps1_a, eps2_a])
p_end   = np.array([eps1_b, eps2_b])

n = 300
t = np.linspace(0, 1, n)
line = p_start + t[:, None] * (p_end - p_start)   # shape (n, 2): col 0 = eps1, col 1 = eps2

# Query order must match the grid: (eps1, eps2)
profile = interp(line)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
ax1.pcolormesh(list_eps1, list_eps2, prevalences, cmap='turbo', shading='auto')
ax1.plot(line[:, 1], line[:, 0], color='magenta', label='Slice')
ax1.set_xlabel(r'$\tilde{\epsilon}_1$')
ax1.set_ylabel(r'$\tilde{\epsilon}_2$')
ax1.set_title('Average Prevalence I(t) over last 10 years')
ax1.legend()


ax2.plot(np.linspace(eps1_a, eps1_b, n), profile, color='magenta',label='Slice' )
ax2.set_xlabel(r'$\tilde{\epsilon}_1$')
ax2.set_ylabel('Average Prevalence I(t)')
ax2.legend()

ax2.grid()
#Fix ratio of axes
ax1.set_aspect('equal', adjustable='box')
plt.tight_layout()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))
ax1.contourf(list_eps1, list_eps2, prevalences, levels=np.logspace(-3.5, -2., 20), cmap='turbo')
ax1.set_xlabel(r'$\tilde{\epsilon}_1$')
ax1.set_ylabel(r'$\tilde{\epsilon}_2$')
ax1.set_title('Average Prevalence I(t) over last 10 years')

#Acrross an approximate level curve
eps1_a, eps2_a = -0.55, 0.5
eps1_b, eps2_b = -0.65, 1.8

# Define the line by two endpoints in (eps1, eps2) space
p_start = np.array([eps1_a, eps2_a])
p_end   = np.array([eps1_b, eps2_b])
n = 300
t = np.linspace(0, 1, n)
line1 = p_start + t[:, None] * (p_end - p_start)   # shape (n, 2): col 0 = eps1, col 1 = eps2
# Query order must match the grid: (eps1, eps2)
profile1 = interp(line1)
ax1.plot(line1[:, 0], line1[:, 1], color='magenta', label='Slice 1')
ax2.plot(np.linspace(eps2_a, eps2_b, n), profile1, color='magenta',label='Slice 1' )


#Across several level curves
eps1_a, eps2_a = 0, 0.5
eps1_b, eps2_b = 0.5, 1.8
# Define the line by two endpoints in (eps1, eps2) space
p_start = np.array([eps1_a, eps2_a])
p_end   = np.array([eps1_b, eps2_b])
n = 300
t = np.linspace(0, 1, n)
line1 = p_start + t[:, None] * (p_end - p_start)   # shape (n, 2): col 0 = eps1, col 1 = eps2
# Query order must match the grid: (eps1, eps2)
profile1 = interp(line1)
ax1.plot(line1[:, 0], line1[:, 1], color='k', label='Slice 2')
ax2.plot(np.linspace(eps2_a, eps2_b, n), profile1, color='k',label='Slice 2' )


ax2.set_xlabel(r'$\tilde{\epsilon}_2$')
ax2.set_ylabel('Average Prevalence I(t)')
ax2.legend()
#Gray background
# ax2.set_facecolor('gray')
ax2.grid()
#Fix ratio of axes

ax1.legend()

ax1.set_aspect('equal', adjustable='box')
plt.tight_layout()
plt.show()
